# Matplotlib: A Hands-On Lesson

Matplotlib is Python's standard plotting library. Pandas, statsmodels, and scikit-learn all draw their charts with it, so learning it pays off everywhere.

This lesson covers:

1. The mental model: figures and axes
2. Line plots
3. Bar charts
4. Histograms (and why claim costs need a log scale)
5. Scatter plots
6. Box plots
7. Multiple plots in one figure (subplots)
8. Styling, labels, and annotations
9. Plotting straight from pandas
10. Saving figures
11. A realistic example: actual vs expected by segment
12. Practice exercises, with solutions

We'll use a simulated set of claims, so there's nothing to download. Run the cells in order.

## 1. The mental model

Almost all confusion with matplotlib comes from not knowing its two main objects:

- **`Figure`**: the whole canvas, like a sheet of paper.
- **`Axes`**: one plot on that canvas, with its own x-axis, y-axis, title, and data. A figure can hold many axes.

(An `Axes` is a *plot*, not a single axis line. The word is confusing.)

There are two ways to use matplotlib:

- **pyplot style** (`plt.plot(...)`): quick, and it keeps track of the "current" plot for you.
- **object-oriented style** (`fig, ax = plt.subplots()` then `ax.plot(...)`): explicit about which plot you're changing.

**Use the object-oriented style.** It's clearer, and it scales to figures with several plots. This lesson uses it throughout.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("matplotlib is ready")

A first plot. `plt.subplots()` creates a figure and one set of axes. We draw on the axes (`ax`):

In [ ]:
fig, ax = plt.subplots()
ax.plot([1, 2, 3, 4], [10, 20, 15, 30])
plt.show()

## 2. Line plots

Line plots suit values that change over a sequence, like months. Here we plot monthly claim totals:

In [ ]:
months = np.arange(1, 13)
claims_total = np.array([120, 135, 128, 150, 162, 158, 171, 180, 165, 190, 205, 220])

fig, ax = plt.subplots()
ax.plot(months, claims_total, marker="o")
ax.set_xlabel("Month")
ax.set_ylabel("Total claims (thousands)")
ax.set_title("Monthly claim totals")
plt.show()

Notice that the labels and title are set with `ax.set_...` methods. In the object-oriented style, you set things on the axes.

You can draw several lines on the same axes. Add `label=` to each line, and `ax.legend()` to display them:

In [ ]:
last_year = claims_total * 0.9

fig, ax = plt.subplots()
ax.plot(months, claims_total, marker="o", label="This year")
ax.plot(months, last_year, marker="s", linestyle="--", label="Last year")
ax.set_xlabel("Month")
ax.set_ylabel("Total claims (thousands)")
ax.legend()
plt.show()

## 3. Bar charts

Bar charts compare values across categories.

In [ ]:
regions = ["Urban", "Suburban", "Rural"]
avg_cost = [4200, 3600, 3100]

fig, ax = plt.subplots()
ax.bar(regions, avg_cost, color="steelblue")
ax.set_ylabel("Average claim cost")
ax.set_title("Average claim cost by region")
plt.show()

For long category names, a horizontal bar chart (`ax.barh`) is easier to read.

## 4. Histograms

A histogram shows how values are distributed. It is usually the first plot to make with a new numeric column.

First we simulate some data to work with. Claim costs are positive and right-skewed, so we use a lognormal distribution:

In [ ]:
rng = np.random.default_rng(seed=1)
n = 3000

df = pd.DataFrame({
    "age": rng.integers(18, 80, size=n),
    "region": rng.choice(["Urban", "Suburban", "Rural"], size=n, p=[0.4, 0.35, 0.25]),
    "smoker": rng.choice(["no", "yes"], size=n, p=[0.8, 0.2]),
})

log_mean = 7.6 + 0.01 * (df["age"] - 40) + 0.35 * (df["smoker"] == "yes")
df["cost"] = rng.lognormal(mean=log_mean, sigma=0.8)

df.head()

In [ ]:
fig, ax = plt.subplots()
ax.hist(df["cost"], bins=40, color="steelblue", edgecolor="white")
ax.set_xlabel("Claim cost")
ax.set_ylabel("Number of claims")
ax.set_title("Distribution of claim costs")
plt.show()

Most claims are small and a few are very large, so the plot is squashed against the left edge. Plotting the **log** of the cost (or using log-spaced bins) shows the shape much better:

In [ ]:
fig, ax = plt.subplots()
ax.hist(np.log10(df["cost"]), bins=40, color="steelblue", edgecolor="white")
ax.set_xlabel("log10 of claim cost")
ax.set_ylabel("Number of claims")
ax.set_title("Claim costs on a log scale")
plt.show()

Always label axes with what was transformed. A reader who sees "log10 of claim cost" knows not to read the numbers as dollars.

## 5. Scatter plots

Scatter plots show the relationship between two numeric variables. With thousands of points, lower the opacity (`alpha`) so overlapping points stay readable:

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df["age"], df["cost"], alpha=0.25, s=10)
ax.set_xlabel("Age")
ax.set_ylabel("Claim cost")
ax.set_yscale("log")           # log axis, because costs are skewed
ax.set_title("Claim cost vs age")
plt.show()

`ax.set_yscale("log")` changes the axis without changing the data, so the tick labels still show real dollar amounts.

Color points by a category with a loop:

In [ ]:
fig, ax = plt.subplots()
for smoker, group in df.groupby("smoker"):
    ax.scatter(group["age"], group["cost"], alpha=0.3, s=10, label=f"smoker: {smoker}")

ax.set_yscale("log")
ax.set_xlabel("Age")
ax.set_ylabel("Claim cost")
ax.legend()
plt.show()

## 6. Box plots

A box plot summarizes a distribution. The box spans the middle 50% of values, the line inside is the median, and the whiskers and dots show the spread and outliers. It's good for comparing groups side by side:

In [ ]:
groups = [df.loc[df["region"] == r, "cost"] for r in ["Urban", "Suburban", "Rural"]]

fig, ax = plt.subplots()
ax.boxplot(groups, tick_labels=["Urban", "Suburban", "Rural"])
ax.set_yscale("log")
ax.set_ylabel("Claim cost")
ax.set_title("Claim cost by region")
plt.show()

## 7. Subplots: several plots in one figure

`plt.subplots(rows, columns)` creates a grid of axes. `axes` is then an array you can index.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(np.log10(df["cost"]), bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Cost distribution (log10)")
axes[0].set_xlabel("log10 of claim cost")

axes[1].scatter(df["age"], df["cost"], alpha=0.2, s=8)
axes[1].set_yscale("log")
axes[1].set_title("Cost vs age")
axes[1].set_xlabel("Age")

fig.suptitle("Claim cost overview")
fig.tight_layout()
plt.show()

`figsize=(width, height)` is in inches. `fig.tight_layout()` stops labels from overlapping, and it's worth adding to any multi-plot figure.

With a 2 × 2 grid, `axes` has two dimensions, so you'd write `axes[0, 1]`. A shortcut is `axes = axes.ravel()` to get a flat list.

## 8. Styling, labels, and annotations

Small changes make a chart readable. Here's a chart with common touches: a reference line, an annotation, and a grid.

In [ ]:
mean_by_age = df.groupby(pd.cut(df["age"], bins=[17, 30, 45, 60, 80]), observed=True)["cost"].mean()
labels = [str(interval) for interval in mean_by_age.index]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, mean_by_age.values, color="#4C78A8")

overall = df["cost"].mean()
ax.axhline(overall, color="crimson", linestyle="--", label=f"Overall mean: {overall:,.0f}")

ax.bar_label(bars, fmt="{:,.0f}", padding=3)
ax.set_xlabel("Age group")
ax.set_ylabel("Average claim cost")
ax.set_title("Average claim cost by age group")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.show()

Handy options:

- `color=`: a name (`"steelblue"`), a hex code (`"#4C78A8"`), or a grayscale string.
- `alpha=`: opacity from 0 (invisible) to 1 (solid).
- `linestyle=`: `"-"`, `"--"`, `":"`.
- `ax.set_xlim(a, b)` and `ax.set_ylim(a, b)`: set the visible range.
- `ax.axhline(y)` and `ax.axvline(x)`: draw reference lines.
- `ax.annotate("text", xy=(x, y))`: point at a spot on the chart.

A built-in style gives the whole figure a consistent look. Try `plt.style.use("ggplot")` and re-run an earlier cell. Available styles are listed in `plt.style.available`.

## 9. Plotting straight from pandas

Pandas DataFrames and Series have a `.plot` method that draws with matplotlib. It's shorter for quick looks, and it returns an `Axes` you can keep customizing:

In [ ]:
ax = df.groupby("region")["cost"].mean().plot(kind="bar", color="steelblue", figsize=(6, 3.5))
ax.set_ylabel("Average claim cost")
ax.set_title("Average cost by region (from pandas)")
plt.show()

## 10. Saving figures

`fig.savefig` writes an image file. Use `dpi=150` or higher for sharp images, and `bbox_inches="tight"` so labels aren't cropped. PNG is good for READMEs, and SVG or PDF is better for documents.

In [ ]:
import tempfile
from pathlib import Path

fig, ax = plt.subplots()
ax.hist(np.log10(df["cost"]), bins=40, color="steelblue", edgecolor="white")
ax.set_title("Claim costs on a log scale")

save_path = Path(tempfile.gettempdir()) / "claim_cost_hist.png"
fig.savefig(save_path, dpi=150, bbox_inches="tight")
print("saved to:", save_path)
plt.show()

For a real project, save figures into a folder such as `figures/` and reference them from your README.

## 11. A realistic example: actual vs expected

Pricing models are checked with **actual-vs-expected** charts. Here we compare average actual claim cost with a simple model's prediction, by age group:

In [ ]:
age_bins = pd.cut(df["age"], bins=[17, 30, 45, 60, 80])

# A deliberately simple "model": the overall average for everyone
df["expected"] = df["cost"].mean()

summary = df.groupby(age_bins, observed=True).agg(
    actual=("cost", "mean"),
    expected=("expected", "mean"),
)
summary.index = [str(i) for i in summary.index]

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(summary))
width = 0.38

ax.bar(x - width / 2, summary["actual"], width, label="Actual", color="#4C78A8")
ax.bar(x + width / 2, summary["expected"], width, label="Expected (flat model)", color="#BAB0AC")

ax.set_xticks(x)
ax.set_xticklabels(summary.index)
ax.set_xlabel("Age group")
ax.set_ylabel("Average claim cost")
ax.set_title("Actual vs expected by age group")
ax.legend()
plt.show()

The flat model misses the rise in cost with age, and the chart shows it immediately. A better model, like the GLMs in the other notebook, would track the actual bars closely.

## 12. Practice exercises

Try each exercise before you look at the solution. Cells marked `# TODO` are for you to fill in.

**Exercise 1.** Make a histogram of `age`, with 15 bins, a title, and labeled axes.

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
fig, ax = plt.subplots()
ax.hist(df["age"], bins=15, color="steelblue", edgecolor="white")
ax.set_xlabel("Age")
ax.set_ylabel("Number of policies")
ax.set_title("Age distribution")
plt.show()
```

</details>

**Exercise 2.** Make a bar chart of the average claim cost for smokers and non-smokers. Add the value above each bar.

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
means = df.groupby("smoker")["cost"].mean()

fig, ax = plt.subplots()
bars = ax.bar(means.index, means.values, color=["#4C78A8", "#E45756"])
ax.bar_label(bars, fmt="{:,.0f}", padding=3)
ax.set_ylabel("Average claim cost")
ax.set_title("Average cost by smoker status")
plt.show()
```

</details>

**Exercise 3.** Make a figure with two side-by-side box plots of claim cost: one for smokers and one for non-smokers. Use a log scale on the y-axis.

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
data = [df.loc[df["smoker"] == s, "cost"] for s in ["no", "yes"]]

fig, ax = plt.subplots()
ax.boxplot(data, tick_labels=["Non-smoker", "Smoker"])
ax.set_yscale("log")
ax.set_ylabel("Claim cost")
ax.set_title("Claim cost by smoker status")
plt.show()
```

</details>

**Exercise 4.** Make a 1 x 2 figure. On the left, a histogram of log10 claim cost for urban policies. On the right, the same for rural policies. Give each plot a title.

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)

for ax, region in zip(axes, ["Urban", "Rural"]):
    subset = df.loc[df["region"] == region, "cost"]
    ax.hist(np.log10(subset), bins=30, color="steelblue", edgecolor="white")
    ax.set_title(region)
    ax.set_xlabel("log10 of claim cost")

axes[0].set_ylabel("Number of claims")
fig.tight_layout()
plt.show()
```

`sharex` and `sharey` give both plots the same axes ranges, so they're easy to compare.

</details>

## Summary

- A **Figure** is the canvas, and **Axes** are the individual plots on it.
- Use **`fig, ax = plt.subplots()`** and call methods on `ax`.
- Pick the chart for the question: lines for sequences, bars for categories, histograms for distributions, scatter for relationships, box plots for comparing groups.
- **Skewed data** (like claim costs) usually needs a log scale.
- **`subplots(rows, cols)`** and `tight_layout()` build multi-plot figures.
- Always label axes and add a title. Save with `fig.savefig(..., dpi=150, bbox_inches="tight")`.